# DCPA Checkpoints 一键安全备份（`checkpoints` -> `checkpoints-ba`）

- 可在 **Colab 免费 CPU 运行时** 单独打开运行，**不会打断** 正在训练的 GPU Notebook。
- 自动挂载 Google Drive，将 `/content/drive/MyDrive/DCPA_Project/checkpoints` 完整复制到 `/content/drive/MyDrive/DCPA_Project/checkpoints-ba`，并对 `manifest.json` 记录的活跃槽位执行 `BLAKE2b` + 字节大小校验。

In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("[Info] 未检测到 google.colab 环境，按本地路径模式执行。")

DRIVE_ROOT = Path("/content/drive/MyDrive/DCPA_Project")
SRC_DIR = DRIVE_ROOT / "checkpoints"
DST_DIR = DRIVE_ROOT / "checkpoints-ba"
CHUNK_BYTES = 4 * 1024 * 1024  # 4 MB sequential buffer for Drive FUSE

if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"未找到源目录: {SRC_DIR}（请确认 Google Drive 已挂载且路径正确）")

DST_DIR.mkdir(parents=True, exist_ok=True)


def blake2b_and_size(path: Path) -> tuple[str, int]:
    h = hashlib.blake2b()
    total = 0
    with path.open("rb") as f:
        while True:
            buf = f.read(CHUNK_BYTES)
            if not buf:
                break
            h.update(buf)
            total += len(buf)
    return h.hexdigest(), total


def copy_with_fsync(src: Path, dst: Path) -> tuple[int, str]:
    dst.parent.mkdir(parents=True, exist_ok=True)
    h = hashlib.blake2b()
    written = 0
    with src.open("rb") as fin, dst.open("wb") as fout:
        while True:
            buf = fin.read(CHUNK_BYTES)
            if not buf:
                break
            fout.write(buf)
            h.update(buf)
            written += len(buf)
        fout.flush()
        os.fsync(fout.fileno())
    return written, h.hexdigest()


# 1. 先读取当前活跃槽位信息，确保优先安全备份当前健康 Checkpoint
manifest_path = SRC_DIR / "manifest.json"
manifest_before = None
active_filename = None
if manifest_path.is_file():
    try:
        manifest_before = json.loads(manifest_path.read_text(encoding="utf-8"))
        active_slot = manifest_before.get("active_slot")
        slots = manifest_before.get("slots") or {}
        if active_slot in slots:
            active_filename = slots[active_slot].get("filename") or f"slot_{active_slot}.pt"
        print(
            f"[Manifest] active_slot={active_slot} ({active_filename}) | "
            f"step={manifest_before.get('step')} | stage={manifest_before.get('stage_id')}"
        )
    except Exception as exc:
        print(f"[Warn] 读取 manifest.json 失败（将继续全量复制）: {exc}")

# 2. 收集源目录中所有非临时文件，把 active_slot 与 manifest 排在最前
all_files = [
    p for p in sorted(SRC_DIR.rglob("*"))
    if p.is_file() and not p.name.endswith(".tmp")
]


def _priority(p: Path) -> tuple[int, str]:
    rel = p.relative_to(SRC_DIR).as_posix()
    if active_filename and rel == active_filename:
        return (0, rel)
    if rel in ("manifest.json", "manifest.bak.json"):
        return (1, rel)
    return (2, rel)


all_files.sort(key=_priority)
print(f"[Backup] {SRC_DIR} -> {DST_DIR} ({len(all_files)} 个文件)", flush=True)

t_all = time.perf_counter()
copied_hashes: dict[str, tuple[int, str]] = {}
for src_file in all_files:
    rel_path = src_file.relative_to(SRC_DIR)
    dst_file = DST_DIR / rel_path
    t0 = time.perf_counter()
    sz, digest = copy_with_fsync(src_file, dst_file)
    copied_hashes[rel_path.as_posix()] = (sz, digest)
    dt = time.perf_counter() - t0
    print(f"  [OK] {rel_path.as_posix():<28s} {sz / (1024 * 1024):8.2f} MB ({dt:.1f}s)", flush=True)

# 3. 校验备份后的活跃槽位完整性（同时防止主训练在复制中途正好切换槽位）
if manifest_before is not None and active_filename and active_filename in copied_hashes:
    exp_size = int(manifest_before.get("file_size", -1))
    exp_hash = manifest_before.get("blake2b")
    dst_active = DST_DIR / active_filename
    actual_hash, actual_size = blake2b_and_size(dst_active)
    if exp_size > 0 and actual_size != exp_size:
        raise RuntimeError(
            f"备份校验失败: {active_filename} 大小不匹配 (expected={exp_size}, actual={actual_size})"
        )
    if exp_hash and actual_hash != exp_hash:
        raise RuntimeError(
            f"备份校验失败: {active_filename} BLAKE2b 不匹配 ({actual_hash[:12]} != {str(exp_hash)[:12]})"
        )
    print(f"[Verify OK] {active_filename} 大小({actual_size:,} B)与 BLAKE2b({actual_hash[:12]}...) 完全一致！")

    if manifest_path.is_file():
        try:
            manifest_after = json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest_after.get("step") != manifest_before.get("step"):
                print(
                    f"[Note] 复制期间主训练已同步新一步 (step {manifest_before.get('step')} -> {manifest_after.get('step')})，"
                    f"但备份中的 {active_filename} 已通过完整哈希校验，可安全用于恢复。"
                )
        except Exception:
            pass

print(f"\n[DONE] 全部备份完成，总耗时 {time.perf_counter() - t_all:.1f}s -> {DST_DIR}")
